# TP 2 - Préparation améliorée de la base RAG

---
## 0. Configuration partagée

Ce cahier construit la **base vectorielle V2** utilisée dans `2_4_rag_assistant_improved.ipynb`.
Contrairement à la V1 (découpage par taille fixe), ici le découpage suit la structure Markdown (`###`) et ajoute un contexte hiérarchique.

À exécuter **avant** `2_4_rag_assistant_improved.ipynb`.

In [ ]:
from shared.config import ROOT_DIR
from shared.rag_utils import (
    rag_load_markdown_documents,
    rag_chunk_markdown_by_headers,
    rag_describe_chunks,
    rag_build_chunk_embeddings,
    rag_index_chunks_chroma,
)

DATA_DIR = ROOT_DIR / "TP2_travel_planner_RAG" / "data"
MARKDOWN_DIR = DATA_DIR / "guides_markdown"
CHROMA_DIR_V2 = DATA_DIR / "chroma_db_rag_v2"

MIN_CHUNK_CHARS = 250
MAX_CHUNK_CHARS = 3000
EMBEDDING_BATCH_SIZE = 16

---
## 1. Charger les documents Markdown

Les guides ont déjà été convertis de PDF vers Markdown et nettoyés.
Concrètement, le Markdown rend le découpage plus fiable: titres explicites (`#`, `##`, `###`), retours ligne propres, et moins d'artefacts parasites.
C'est cette structure qui permet le découpage par en-têtes.

In [ ]:
documents = rag_load_markdown_documents(MARKDOWN_DIR)

total_characters = sum(len(doc["text"]) for doc in documents)
print(f"Documents Markdown chargés : {len(documents)}")
print(f"Nombre total de caractères : {total_characters}\n")
for doc in documents:
    print(f"  {doc['source']}: {len(doc['text'])} caractères")

### Inspection: structure Markdown

Vérifier la hiérarchie des titres d'un document.
Le chunker découpe sur `###` et remonte les parents `#`/`##` pour construire le préfixe de contexte.

In [ ]:
sample_doc = documents[0]
print(f"Document : {sample_doc['source']}")
print(f"Total : {len(sample_doc['text'])} caractères\n")
print(sample_doc["text"][:800])

---
## 2. Stratégie de découpage V2 — basée sur les en-têtes Markdown

**TODO — `rag_chunk_markdown_by_headers`**

Fichier à modifier : `shared/rag_utils.py`

La V1 découpe uniquement par taille. La V2 utilise la structure Markdown pour conserver le sens des sections

`rag_chunk_markdown_by_headers` : fonction qui découpe les `MarkdownDocument` selon les en-têtes Markdown et retourne des `RAGChunk` contextualisés (document, section, sous-section)

Découpage attendu : créer les chunks sur les frontières `###`, puis ajouter au début de chaque chunk un préfixe de contexte avec le document et les sections parentes (`#` et `##`)

Paramètres déjà définis : `MIN_CHUNK_CHARS`, `MAX_CHUNK_CHARS`
Après exécution, vérifie dans la cellule d'inspection que le préfixe de contexte est bien présent et correct

Pourquoi c'est utile : un chunk qui contient son contexte de document et de section est mieux compris pendant la recherche que du texte isolé


In [ ]:
# TODO : découper selon la structure markdown et préserver le contexte hiérarchique
chunks = rag_chunk_markdown_by_headers(
    documents=documents,
    min_chunk_chars=MIN_CHUNK_CHARS,
    max_chunk_chars=MAX_CHUNK_CHARS,
)
rag_describe_chunks(chunks)

### Inspection: chunks avec préfixe de contexte

Chaque chunk doit commencer par un en-tête de contexte (`Document / Section / Sous-section / Paragraphe`) avant le contenu.
Comparer visuellement avec un chunk V1 (`2_1`) pour vérifier le gain de contexte.

In [ ]:
sample_indices = [0, len(chunks) // 6, len(chunks) // 5, len(chunks) // 4, len(chunks) // 2, 3 * len(chunks) // 4, len(chunks) - 1]
for idx in sample_indices:
    chunk = chunks[idx]
    print(f"--- Chunk #{chunk.chunk_id} | source: {chunk.source} | {chunk.char_count} caractères ---")
    print(chunk.text)
    print("\n"*5)

---
## 3. Vectorisation et indexation

In [ ]:
chunk_embeddings = rag_build_chunk_embeddings(chunks=chunks, batch_size=EMBEDDING_BATCH_SIZE)

In [ ]:
rag_index_chunks_chroma(persist_dir=CHROMA_DIR_V2, chunks=chunk_embeddings)

print(f"V2 — Chunks indexés : {len(chunk_embeddings)}")
print(f"V2 — Dimension des embeddings : {len(chunk_embeddings[0].embedding)}")
